# GateKeep · notebook 03 · gate training data and the first per-gate results

**Before you start:** attach `gatekeep_data` (it must hold the backup from the last run: the folders `data/`, `cache/`, `results/`, plus the PDF) · Internet **On** · Accelerator **None** · Secret `OLLAMA_API_KEY` on.

Run steps 1-2 interactively, then **Save Version → Save & Run All (Commit)**. Expect about **2-3 hours** and **~14% of the monthly quota**. The run stops at once if no backup is found.

## 1 · code, packages, key

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install docling sentence-transformers faiss-cpu pymupdf pytest openai

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore the backup (stops if there is none)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

In [ ]:
!ls data cache results && wc -l data/qa_*.jsonl

## 3 · build the gate training data (~5,000 calls, 1.5-2 h)
The label balance is printed per gate and split: paste it back. Gate 3 uses Super to grade Gemma's real answers on dev/test.

In [ ]:
!python -m gatekeep.cli gatedata

## 4 · how good are Gemma's gates and the simple classifier? (~3,000 Gemma calls, ~1 h)
Paste back every line. For the *grade* gate look at **let-through**: if Gemma lets only a fraction of the relevant passages through, its prompt is too strict. The retrieval line says whether the passages were even found.

In [ ]:
!python -m gatekeep.cli evalgates llm,sklearn

## 5 · back up, then stop

In [ ]:
!cd /kaggle/working/GateKeep && zip -rq /kaggle/working/gatekeep_backup.zip data cache results && ls -lh /kaggle/working/gatekeep_backup.zip

Download `gatekeep_backup.zip` from the **Output** panel and replace the old backup in your dataset (new version). Keep it outside the git folder.